# OSRM Under the Hood: Road Networks, Graphs, and Contraction Hierarchies

> **Attribution**: [OSRM (Project-OSRM/osrm-backend)](https://github.com/Project-OSRM/osrm-backend) is a C++ routing engine built by 180+ contributors, originally from the University of Karlsruhe. This notebook explains the mathematical foundations of what OSRM implements internally. The Python code here is a pedagogical reimplementation — none of it comes from OSRM's source.

## What this notebook covers

1. **Road networks as directed weighted graphs** — the formal model
2. **Dijkstra's algorithm** — the classic shortest-path algorithm, visualized step by step
3. **The scale problem** — why Dijkstra alone cannot power real-time routing at country scale
4. **Contraction Hierarchies** — OSRM's core preprocessing technique (Geisberger et al., 2008)
5. **Bidirectional CH search** — how queries are answered in microseconds at runtime
6. **Connecting to OSRM** — what `osrm-contract` and `osrm-routed` actually do

**Reference**: Geisberger, R., Sanders, P., Schultes, D., & Delling, D. (2008). *Contraction Hierarchies: Faster and Simpler Hierarchical Routing in Road Networks*. In WEA 2008.

In [ ]:
import heapq
import time
import warnings

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

# Colour palette
C = {
    'unseen':   '#CFD8DC',
    'frontier': '#FFB300',
    'settled':  '#1E88E5',
    'source':   '#43A047',
    'target':   '#E53935',
    'shortcut': '#AB47BC',
    'edge':     '#90A4AE',
    'path':     '#E53935',
}

print('Libraries loaded.')

---
## 1. Road Networks as Directed Weighted Graphs

Every routing engine — OSRM, Google Maps, Mapbox — begins with the same abstraction:

$$G = (V, E, w)$$

| Symbol | Real-world meaning |
|--------|--------------------|
| $V$ | Intersections (nodes) |
| $E \subseteq V \times V$ | Road segments (directed edges) |
| $w: E \to \mathbb{R}^+$ | Edge weight — typically **travel time** in seconds |

Edges are **directed** because roads are not always bidirectional (one-way streets, highway on-ramps). Edge weights encode speed limits, road type, and turn penalties from the `car.lua` profile — not straight-line distance.

The shortest path problem: given source $s$ and target $t$, find the path $p = (s, v_1, v_2, ..., t)$ that minimises $\sum_{(u,v) \in p} w(u, v)$.

In [ ]:
# Build a 10-node toy city graph
# Nodes represent intersections; edge weights are travel time in minutes

G = nx.DiGraph()

# Node positions for layout (x, y)
pos = {
    0: (1, 4),   # North gate
    1: (0, 3),   # NW residential
    2: (2, 3),   # N commercial
    3: (4, 3),   # NE district
    4: (0, 2),   # W avenue
    5: (2, 2),   # City centre
    6: (4, 2),   # E avenue
    7: (0, 1),   # SW residential
    8: (2, 1),   # S commercial
    9: (4, 1),   # SE exit
}

node_labels = {
    0: 'Gate N', 1: 'NW', 2: 'N Com', 3: 'NE',
    4: 'W Ave', 5: 'Centre', 6: 'E Ave',
    7: 'SW', 8: 'S Com', 9: 'SE Exit',
}

G.add_nodes_from(pos.keys())
nx.set_node_attributes(G, pos, 'pos')

# Directed edges: (from, to, travel_time_minutes)
edges = [
    (0, 1, 3), (0, 2, 2),
    (1, 4, 3), (1, 5, 5),
    (2, 1, 2), (2, 5, 3), (2, 3, 4),
    (3, 6, 3),
    (4, 5, 3), (4, 7, 4),
    (5, 4, 3), (5, 6, 3), (5, 8, 4),
    (6, 5, 3), (6, 9, 5),
    (7, 8, 2),
    (8, 7, 2), (8, 9, 3),
    (9, 6, 5),
]
G.add_weighted_edges_from(edges)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# --- Draw ---
fig, ax = plt.subplots(figsize=(10, 7))

nx.draw_networkx_nodes(G, pos, ax=ax, node_size=700,
                       node_color=C['unseen'], edgecolors='#607D8B', linewidths=1.5)
nx.draw_networkx_labels(G, pos, labels=node_labels, ax=ax, font_size=8, font_weight='bold')
nx.draw_networkx_edges(G, pos, ax=ax, edge_color=C['edge'],
                       arrows=True, arrowsize=18, arrowstyle='-|>',
                       connectionstyle='arc3,rad=0.1', width=1.5)
edge_labels = nx.get_edge_attributes(G, 'weight')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels,
                             ax=ax, font_size=7, label_pos=0.35)

ax.set_title('Toy City Road Network\nNodes = intersections · Edge weights = travel time (min)',
             fontsize=13, pad=12)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 2. Dijkstra's Algorithm

Dijkstra (1959) solves the single-source shortest path problem on non-negative weighted graphs. It uses a **min-heap priority queue** to always expand the nearest unvisited node.

**Algorithm:**

```
dist[s] ← 0;  dist[v] ← ∞  for all v ≠ s
pq ← {(0, s)}

while pq is not empty:
    (d, u) ← pop_min(pq)
    if u already settled: skip
    mark u as settled
    for each neighbour v of u:
        if dist[u] + w(u,v) < dist[v]:
            dist[v] ← dist[u] + w(u,v)
            push (dist[v], v) onto pq
```

**Time complexity**: $O((|V| + |E|) \log |V|)$ with a binary heap.

The next cell implements Dijkstra while recording each step so we can visualise the expanding frontier.

In [ ]:
def dijkstra(graph, source):
    """
    Dijkstra's algorithm with step recording for visualisation.
    Returns dist dict, prev dict, and a list of snapshots.
    Each snapshot: {settled, current, dist, frontier}
    """
    dist = {n: float('inf') for n in graph.nodes()}
    prev = {n: None for n in graph.nodes()}
    dist[source] = 0
    pq = [(0, source)]
    settled = set()
    steps = []

    while pq:
        d, u = heapq.heappop(pq)
        if u in settled:
            continue
        settled.add(u)

        steps.append({
            'settled': settled.copy(),
            'current': u,
            'dist': dist.copy(),
            'frontier': {n for (_, n) in pq if n not in settled},
        })

        for v in graph.successors(u):
            w = graph[u][v]['weight']
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(pq, (dist[v], v))

    return dist, prev, steps


def reconstruct_path(prev, source, target):
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = prev[node]
    path.reverse()
    return path if path[0] == source else []


SOURCE, TARGET = 0, 9
dist, prev, steps = dijkstra(G, SOURCE)
shortest_path = reconstruct_path(prev, SOURCE, TARGET)

print(f'Shortest path from node {SOURCE} to node {TARGET}:')
print(f'  Path   : {" → ".join(str(n) for n in shortest_path)}')
print(f'  Time   : {dist[TARGET]} min')
print(f'  Steps  : {len(steps)} node settlements')

In [ ]:
def draw_step(ax, graph, step, path_edges, step_num, total_steps):
    """Draw a single Dijkstra step on an axes object."""
    node_colors = []
    for n in graph.nodes():
        if n == SOURCE:
            node_colors.append(C['source'])
        elif n == TARGET:
            node_colors.append(C['target'])
        elif n in step['settled']:
            node_colors.append(C['settled'])
        elif n in step['frontier']:
            node_colors.append(C['frontier'])
        else:
            node_colors.append(C['unseen'])

    nx.draw_networkx_nodes(graph, pos, ax=ax, node_size=500,
                           node_color=node_colors, edgecolors='#455A64', linewidths=1)

    # Edge colours: highlight path edges
    edge_colors = [C['path'] if e in path_edges else C['edge'] for e in graph.edges()]
    edge_widths = [3 if e in path_edges else 1 for e in graph.edges()]
    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color=edge_colors,
                           width=edge_widths, arrows=True, arrowsize=14,
                           arrowstyle='-|>', connectionstyle='arc3,rad=0.1')

    # Distance labels
    dist_labels = {
        n: (f"{step['dist'][n]:.0f}" if step['dist'][n] != float('inf') else '∞')
        for n in graph.nodes()
    }
    shifted = {k: (v[0], v[1] + 0.2) for k, v in pos.items()}
    nx.draw_networkx_labels(graph, shifted, labels=dist_labels, ax=ax,
                            font_size=7, font_color='#212121')

    ax.set_title(f'Step {step_num}/{total_steps}: settling node {step["current"]}\n'
                 f'dist={step["dist"][step["current"]]:.0f} min', fontsize=9)
    ax.axis('off')


# Path edge set for highlighting
path_edges = set(zip(shortest_path[:-1], shortest_path[1:]))

# Pick 4 representative steps
indices = [0, len(steps) // 3, 2 * len(steps) // 3, len(steps) - 1]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, i in zip(axes.flatten(), indices):
    draw_step(ax, G, steps[i], path_edges, i + 1, len(steps))

# Legend
legend_elements = [
    mpatches.Patch(color=C['source'],   label='Source (node 0)'),
    mpatches.Patch(color=C['target'],   label='Target (node 9)'),
    mpatches.Patch(color=C['settled'],  label='Settled — optimal distance found'),
    mpatches.Patch(color=C['frontier'], label='Frontier — in priority queue'),
    mpatches.Patch(color=C['unseen'],   label='Unseen'),
    mpatches.Patch(color=C['path'],     label='Shortest path'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

fig.suptitle("Dijkstra's Algorithm — Step by Step\n"
             "Numbers on nodes = tentative distance from source (minutes)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. The Scale Problem

On our 10-node toy city, Dijkstra is instantaneous. But real road networks are massive:

| Region | Nodes (approx.) | Edges (approx.) |
|--------|-----------------|-----------------|
| Caracas metro | ~200,000 | ~500,000 |
| Venezuela | ~1,500,000 | ~4,000,000 |
| South America | ~50,000,000 | ~130,000,000 |

Dijkstra's complexity is $O((|V| + |E|) \log |V|)$. Let's measure how it scales.

In [ ]:
def time_dijkstra(n_nodes):
    """Create a random road-like graph and time a single-source Dijkstra query."""
    # Erdos-Renyi graph with road-network-like sparsity (~3 edges per node)
    p = min(3 / n_nodes, 1.0)
    rng = np.random.default_rng(42)
    H = nx.gnp_random_graph(n_nodes, p, directed=True, seed=42)
    for u, v in H.edges():
        H[u][v]['weight'] = rng.uniform(1, 10)

    source = 0
    start = time.perf_counter()
    nx.single_source_dijkstra(H, source, weight='weight')
    elapsed = time.perf_counter() - start
    return elapsed


sizes = [100, 500, 1_000, 5_000, 10_000, 50_000]
times = [time_dijkstra(n) for n in sizes]

# Extrapolate to real-world sizes
from numpy.polynomial import polynomial as P
log_sizes = np.log10(sizes)
log_times = np.log10(times)
coeffs = np.polyfit(log_sizes, log_times, 1)

extra_sizes = [200_000, 1_500_000]
extra_times = [10 ** np.polyval(coeffs, np.log10(n)) for n in extra_sizes]

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
all_s = sizes + extra_sizes
all_t = times + extra_times

ax.loglog(sizes, [t * 1000 for t in times], 'o-',
          color=C['settled'], linewidth=2, markersize=7, label='Measured')
ax.loglog(extra_sizes, [t * 1000 for t in extra_times], 's--',
          color=C['target'], linewidth=1.5, markersize=7, label='Extrapolated')

# Annotations
ax.axvline(200_000, color='#78909C', linestyle=':', linewidth=1)
ax.axvline(1_500_000, color='#78909C', linestyle=':', linewidth=1)
ax.text(200_000 * 1.1, ax.get_ylim()[0] * 2, 'Caracas\nmetro', fontsize=8, color='#546E7A')
ax.text(1_500_000 * 1.1, ax.get_ylim()[0] * 2, 'Venezuela', fontsize=8, color='#546E7A')

ax.set_xlabel('Graph size (nodes)', fontsize=11)
ax.set_ylabel('Query time (ms)', fontsize=11)
ax.set_title('Dijkstra Query Time vs. Graph Size\n'
             'At Venezuela scale, a single query takes seconds', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Extrapolated Dijkstra time at Venezuela scale (~1.5M nodes): '
      f'{extra_times[1]*1000:.0f} ms')
print('At 2M daily queries that would require enormous compute — and all of it on the hot path.')

---
## 4. Contraction Hierarchies — The Core Idea

**Reference**: Geisberger et al. (2008)

The fundamental insight: **not all nodes are equally important**.

A rural dirt-track intersection is less important than a highway interchange. If we could remove less important nodes and preserve all shortest paths, we would get a sparser, faster-to-search graph.

CH does exactly this in two phases:

### Preprocessing (offline — runs as `osrm-contract`)

1. Assign an **importance rank** to every node
2. **Contract** nodes in order from least to most important:
   - Remove node $v$
   - For every pair $(u, w)$ where $u \to v \to w$ was the unique shortest path, add a **shortcut edge** $u \to w$ with $w(u,w) = w(u,v) + w(v,w)$
3. The result is the original graph plus shortcut edges, where each node has a rank

### Node importance metrics

OSRM uses a combination of:
- **Edge difference**: shortcuts added − edges removed. Lower = cheaper to contract.
- **Deleted neighbours**: how many already-contracted neighbours. Lower = better.
- **Search space size**: how large the local Dijkstra space is. Smaller = less important.

### Query (online — runs inside `osrm-routed`)

Bidirectional Dijkstra where each side **only relaxes upward edges** (towards higher-ranked nodes). The two searches meet near the top. Instead of exploring millions of nodes, a query touches only a few hundred.

In [ ]:
def edge_difference(graph, node):
    """
    Compute edge difference for a node (a key node importance metric).
    edge_difference = shortcuts_needed - edges_incident
    Lower values → cheaper to contract → lower importance rank.
    """
    preds = list(graph.predecessors(node))
    succs = list(graph.successors(node))
    edges_incident = len(preds) + len(succs)

    shortcuts_needed = 0
    H = graph.copy()
    H.remove_node(node)

    for u in preds:
        for w in succs:
            if u == w:
                continue
            path_through = graph[u][node]['weight'] + graph[node][w]['weight']
            try:
                alt = nx.shortest_path_length(H, u, w, weight='weight')
                if alt > path_through:
                    shortcuts_needed += 1
            except nx.NetworkXNoPath:
                shortcuts_needed += 1

    return shortcuts_needed - edges_incident


# Compute importance for all nodes
importance = {n: edge_difference(G, n) for n in G.nodes()}
rank_order = sorted(importance, key=lambda n: importance[n])

print('Node importance (edge difference) — lower = less important = contracted first:')
for n in rank_order:
    label = node_labels[n]
    print(f'  Node {n:2d} ({label:<12s}): edge_diff = {importance[n]:+d}')

In [ ]:
def contract_node(graph, node):
    """
    Contract a single node:
      - For every (u → node → w) where this is the shortest path,
        add shortcut u → w.
      - Remove the node.
    Returns (new_graph, shortcuts_added)
    """
    G_new = graph.copy()
    preds = list(graph.predecessors(node))
    succs = list(graph.successors(node))
    shortcuts = []

    G_without = graph.copy()
    G_without.remove_node(node)

    for u in preds:
        for w in succs:
            if u == w:
                continue
            path_through = graph[u][node]['weight'] + graph[node][w]['weight']
            try:
                alt = nx.shortest_path_length(G_without, u, w, weight='weight')
                if alt <= path_through:
                    continue
            except nx.NetworkXNoPath:
                pass

            if G_new.has_edge(u, w):
                if G_new[u][w]['weight'] > path_through:
                    G_new[u][w]['weight'] = path_through
                    G_new[u][w]['shortcut'] = True
                    shortcuts.append((u, w, path_through))
            else:
                G_new.add_edge(u, w, weight=path_through, shortcut=True)
                shortcuts.append((u, w, path_through))

    G_new.remove_node(node)
    return G_new, shortcuts


# Visualise contracting the least important node
node_to_contract = rank_order[0]
G_contracted, shortcuts = contract_node(G, node_to_contract)

print(f'Contracting node {node_to_contract} ({node_labels[node_to_contract]})')
print(f'Shortcuts added: {len(shortcuts)}')
for u, w, wt in shortcuts:
    print(f'  Shortcut: {u} ({node_labels[u]}) → {w} ({node_labels[w]}), weight={wt}')

# --- Draw before / after ---
pos_remaining = {k: v for k, v in pos.items() if k != node_to_contract}
labels_remaining = {k: v for k, v in node_labels.items() if k != node_to_contract}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Before
node_cols_before = [C['target'] if n == node_to_contract else C['unseen'] for n in G.nodes()]
nx.draw_networkx_nodes(G, pos, ax=ax1, node_size=600, node_color=node_cols_before,
                       edgecolors='#607D8B', linewidths=1.5)
nx.draw_networkx_labels(G, pos, labels=node_labels, ax=ax1, font_size=8, font_weight='bold')
nx.draw_networkx_edges(G, pos, ax=ax1, edge_color=C['edge'], arrows=True,
                       arrowsize=16, arrowstyle='-|>', connectionstyle='arc3,rad=0.1', width=1.5)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, 'weight'),
                             ax=ax1, font_size=7, label_pos=0.35)
ax1.set_title(f'Before: node {node_to_contract} ({node_labels[node_to_contract]}) highlighted\n'
              f'(edge_diff = {importance[node_to_contract]:+d}, contracted first)',
              fontsize=11)
ax1.axis('off')

# After
shortcut_edges = {(u, w) for u, w, _ in shortcuts}
edge_cols = [C['shortcut'] if e in shortcut_edges else C['edge']
             for e in G_contracted.edges()]
edge_widths = [2.5 if e in shortcut_edges else 1.2 for e in G_contracted.edges()]

nx.draw_networkx_nodes(G_contracted, pos_remaining, ax=ax2, node_size=600,
                       node_color=C['unseen'], edgecolors='#607D8B', linewidths=1.5)
nx.draw_networkx_labels(G_contracted, pos_remaining, labels=labels_remaining,
                        ax=ax2, font_size=8, font_weight='bold')
nx.draw_networkx_edges(G_contracted, pos_remaining, ax=ax2,
                       edge_color=edge_cols, width=edge_widths, arrows=True,
                       arrowsize=16, arrowstyle='-|>', connectionstyle='arc3,rad=0.1')
new_edge_labels = nx.get_edge_attributes(G_contracted, 'weight')
nx.draw_networkx_edge_labels(G_contracted, pos_remaining, edge_labels=new_edge_labels,
                             ax=ax2, font_size=7, label_pos=0.35)
ax2.set_title(f'After: node {node_to_contract} removed\n'
              f'Purple edges = shortcuts preserving shortest paths', fontsize=11)
ax2.axis('off')

# Legend
legend_els = [
    mpatches.Patch(color=C['target'], label=f'Node being contracted'),
    mpatches.Patch(color=C['shortcut'], label='Shortcut edges (new)'),
    mpatches.Patch(color=C['edge'], label='Original edges'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=3, fontsize=9,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Node Contraction: Removing a Node and Adding Shortcut Edges',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Building the Full Hierarchy

In a real CH build, all nodes are contracted in importance order. After the full contraction, every node has a **rank** (its contraction order). This forms the hierarchy:

- **Low-rank nodes**: contracted early — local streets, minor roads
- **High-rank nodes**: contracted last — highway interchanges, major arterials

OSRM's `osrm-contract` runs this process on millions of nodes and produces the `.osrm.hsgr` file.

In [ ]:
# Build the full CH hierarchy on our toy graph
# Track the contraction order → this becomes the node rank

def build_ch(graph):
    """
    Full CH construction: contract all nodes in order of edge difference.
    Returns:
      ch_graph: augmented graph with shortcut edges
      rank: dict {node: rank} (higher rank = more important)
    """
    remaining = graph.copy()
    ch_graph = graph.copy()
    rank = {}
    contraction_order = []

    current_rank = 0
    nodes_left = set(remaining.nodes())

    while nodes_left:
        # Find least important node in remaining graph
        imp = {n: edge_difference(remaining, n) for n in nodes_left}
        node = min(imp, key=lambda n: imp[n])

        rank[node] = current_rank
        contraction_order.append(node)
        current_rank += 1

        # Contract in remaining (to update witness searches)
        remaining_new, shortcuts = contract_node(remaining, node)
        # Add shortcuts to ch_graph too
        for u, w, wt in shortcuts:
            if ch_graph.has_edge(u, w):
                if ch_graph[u][w]['weight'] > wt:
                    ch_graph[u][w].update({'weight': wt, 'shortcut': True})
            else:
                ch_graph.add_edge(u, w, weight=wt, shortcut=True)

        remaining = remaining_new
        nodes_left.remove(node)

    return ch_graph, rank, contraction_order


ch_graph, node_rank, contraction_order = build_ch(G)

print('Contraction order (least → most important):')
for i, n in enumerate(contraction_order):
    print(f'  Rank {i:2d}: node {n} ({node_labels[n]})')

n_original = G.number_of_edges()
n_shortcuts = sum(1 for u, v in ch_graph.edges() if ch_graph[u][v].get('shortcut', False))
print(f'\nOriginal edges : {n_original}')
print(f'Shortcut edges : {n_shortcuts}')
print(f'Total CH edges : {ch_graph.number_of_edges()}')

In [ ]:
# Visualise the hierarchy — colour nodes by rank

ranks = np.array([node_rank[n] for n in ch_graph.nodes()])
rank_normalised = ranks / ranks.max()

cmap = plt.cm.RdYlGn  # Red (low rank / less important) → Green (high rank / more important)
node_colors_rank = [cmap(r) for r in rank_normalised]

fig, ax = plt.subplots(figsize=(11, 7))

sc = nx.draw_networkx_nodes(ch_graph, pos, ax=ax, node_size=700,
                             node_color=rank_normalised, cmap=cmap,
                             edgecolors='#455A64', linewidths=1.5,
                             vmin=0, vmax=1)

rank_labels = {n: f'{node_labels[n]}\nr{node_rank[n]}' for n in ch_graph.nodes()}
nx.draw_networkx_labels(ch_graph, pos, labels=rank_labels, ax=ax, font_size=7)

edge_cols = [C['shortcut'] if ch_graph[u][v].get('shortcut') else C['edge']
             for u, v in ch_graph.edges()]
edge_widths = [2.5 if ch_graph[u][v].get('shortcut') else 1 for u, v in ch_graph.edges()]

nx.draw_networkx_edges(ch_graph, pos, ax=ax, edge_color=edge_cols,
                       width=edge_widths, arrows=True, arrowsize=16,
                       arrowstyle='-|>', connectionstyle='arc3,rad=0.1')

plt.colorbar(plt.cm.ScalarMappable(cmap=cmap), ax=ax,
             label='Node rank (0 = contracted first / least important, 1 = most important)',
             fraction=0.03)

legend_els = [
    mpatches.Patch(color=C['shortcut'], label='Shortcut edges'),
    mpatches.Patch(color=C['edge'], label='Original edges'),
]
ax.legend(handles=legend_els, loc='upper right', fontsize=9)

ax.set_title('Contraction Hierarchy — Augmented Graph with Node Ranks\n'
             'r0 = first contracted (low importance), r9 = last (high importance)',
             fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 6. CH Bidirectional Query — Why It's Fast

At query time, `osrm-routed` runs a **bidirectional Dijkstra on the CH graph** with one critical constraint:

> Each side of the search only relaxes edges pointing **upward** in the hierarchy (toward higher-ranked nodes).

**Forward search** (from source): explores upward-rank edges  
**Backward search** (from target): explores upward-rank edges in the reversed graph  

The two searches meet near the top of the hierarchy (at high-rank nodes). Because most nodes are low-rank and are quickly pruned, the search space collapses from millions of nodes to **hundreds**.

The meeting point candidate with minimum $d_{forward}(v) + d_{backward}(v)$ gives the shortest path.

In [ ]:
def ch_query(ch_graph, node_rank, source, target):
    """
    CH bidirectional Dijkstra: each side only goes upward in rank.
    Returns shortest distance and nodes touched by each search.
    """
    # Forward: only upward edges (to higher-rank nodes)
    def forward_step(u, graph, dist, pq, settled):
        for v in graph.successors(u):
            if node_rank[v] > node_rank[u]:  # upward only
                w = graph[u][v]['weight']
                if dist[u] + w < dist.get(v, float('inf')):
                    dist[v] = dist[u] + w
                    heapq.heappush(pq, (dist[v], v))

    # Backward: reversed graph, only upward edges
    reversed_ch = ch_graph.reverse()

    dist_fwd = {source: 0.0}
    dist_bwd = {target: 0.0}
    pq_fwd = [(0.0, source)]
    pq_bwd = [(0.0, target)]
    settled_fwd, settled_bwd = set(), set()
    touched_fwd, touched_bwd = set(), set()

    best = float('inf')
    meeting_node = None

    while pq_fwd or pq_bwd:
        # Forward step
        if pq_fwd:
            d, u = heapq.heappop(pq_fwd)
            if u not in settled_fwd:
                settled_fwd.add(u)
                touched_fwd.add(u)
                forward_step(u, ch_graph, dist_fwd, pq_fwd, settled_fwd)
                if u in dist_bwd:
                    candidate = dist_fwd[u] + dist_bwd[u]
                    if candidate < best:
                        best, meeting_node = candidate, u

        # Backward step (on reversed graph)
        if pq_bwd:
            d, u = heapq.heappop(pq_bwd)
            if u not in settled_bwd:
                settled_bwd.add(u)
                touched_bwd.add(u)
                forward_step(u, reversed_ch, dist_bwd, pq_bwd, settled_bwd)
                if u in dist_fwd:
                    candidate = dist_fwd[u] + dist_bwd[u]
                    if candidate < best:
                        best, meeting_node = candidate, u

    return best, touched_fwd, touched_bwd, meeting_node


ch_dist, touched_fwd, touched_bwd, meeting = ch_query(ch_graph, node_rank, SOURCE, TARGET)
standard_dist = dist[TARGET]  # from the earlier plain Dijkstra

print(f'Standard Dijkstra distance  : {standard_dist} min')
print(f'CH bidirectional distance   : {ch_dist:.1f} min')
print(f'Results match               : {abs(ch_dist - standard_dist) < 0.01}')
print()
print(f'Nodes touched by standard Dijkstra : {len(steps[-1]["settled"])} / {G.number_of_nodes()}')
print(f'Nodes touched by CH (fwd + bwd)    : {len(touched_fwd) + len(touched_bwd)} / {G.number_of_nodes()}')
print(f'Meeting node                       : {meeting} ({node_labels[meeting]})')

In [ ]:
# Visualise the CH search: which nodes did each side touch?

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

def draw_ch_search(ax, graph, touched_set, other_touched, meeting_node,
                   source_node, target_node, color, title):
    node_cols = []
    for n in graph.nodes():
        if n == source_node:
            node_cols.append(C['source'])
        elif n == target_node:
            node_cols.append(C['target'])
        elif n == meeting_node:
            node_cols.append('#FF6F00')  # amber = meeting point
        elif n in touched_set:
            node_cols.append(color)
        else:
            node_cols.append(C['unseen'])

    nx.draw_networkx_nodes(graph, pos, ax=ax, node_size=650,
                           node_color=node_cols, edgecolors='#455A64', linewidths=1.5)
    nx.draw_networkx_labels(graph, pos,
                            labels={n: f'{node_labels[n]}\nr{node_rank[n]}' for n in graph.nodes()},
                            ax=ax, font_size=7)

    edge_cols = [C['shortcut'] if graph[u][v].get('shortcut') else C['edge']
                 for u, v in graph.edges()]
    nx.draw_networkx_edges(graph, pos, ax=ax, edge_color=edge_cols,
                           width=1.2, arrows=True, arrowsize=14,
                           arrowstyle='-|>', connectionstyle='arc3,rad=0.1')
    ax.set_title(title, fontsize=10)
    ax.axis('off')

draw_ch_search(ax1, ch_graph, touched_fwd, touched_bwd, meeting,
               SOURCE, TARGET, C['settled'],
               f'Forward search from node {SOURCE}\n'
               f'Only upward edges · {len(touched_fwd)} nodes touched')

draw_ch_search(ax2, ch_graph, touched_bwd, touched_fwd, meeting,
               TARGET, SOURCE, '#7B1FA2',
               f'Backward search from node {TARGET}\n'
               f'Only upward edges · {len(touched_bwd)} nodes touched')

legend_els = [
    mpatches.Patch(color=C['source'],   label=f'Source (node {SOURCE})'),
    mpatches.Patch(color=C['target'],   label=f'Target (node {TARGET})'),
    mpatches.Patch(color='#FF6F00',     label='Meeting point'),
    mpatches.Patch(color=C['settled'],  label='Forward search space'),
    mpatches.Patch(color='#7B1FA2',     label='Backward search space'),
    mpatches.Patch(color=C['unseen'],   label='Not touched (pruned)'),
    mpatches.Patch(color=C['shortcut'], label='Shortcut edges'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=4, fontsize=9,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.02))

fig.suptitle('CH Bidirectional Search\n'
             'Each side only expands upward in the hierarchy — search spaces meet near the top',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Connecting This to OSRM

Everything shown in this notebook maps directly to OSRM's command-line pipeline:

| What this notebook does | What OSRM does |
|---|---|
| `G = nx.DiGraph()` with OSM-style edges | `osrm-extract -p /opt/car.lua venezuela.osm.pbf` |
| `edge_difference(G, node)` for each node | Internal node ordering in `osrm-contract` |
| `contract_node(G, node)` iteratively | `osrm-contract venezuela.osrm` → produces `.osrm.hsgr` |
| `ch_query(ch_graph, rank, source, target)` | Every HTTP request to `osrm-routed --algorithm ch` |

The key difference: OSRM's C++ implementation runs this on **millions of nodes in minutes** during preprocessing, then answers queries in **microseconds** at runtime. The Python code here is a pedagogical reimplementation — the algorithm is identical, the scale is not.

### Why this matters for distance matrices

A distance matrix of $N$ origins × $M$ destinations requires $N \times M$ individual queries. At 10 Venezuelan cities that is 100 queries. At 1,000 trip waypoints per day, it is 1,000,000 queries.

OSRM's `/table` endpoint handles all $N \times M$ pairs in a single optimised request using a many-to-many CH variant — far more efficient than calling `/route` $N \times M$ times.

---
## Summary

1. **Road networks are directed weighted graphs** $G = (V, E, w)$ where $w$ encodes travel time
2. **Dijkstra** solves shortest paths optimally but is $O((|V|+|E|)\log|V|)$ — too slow at country scale for real-time use
3. **Contraction Hierarchies** (Geisberger et al., 2008) preprocess the graph once offline:
   - Rank all nodes by importance
   - Contract least important nodes first, adding shortcut edges
   - The result is a hierarchy of nodes from local streets to motorways
4. **CH queries** use bidirectional Dijkstra that only expands upward — touching hundreds of nodes instead of millions
5. **OSRM** implements all of this in C++: `osrm-extract` builds the graph, `osrm-contract` builds the hierarchy, `osrm-routed` serves queries

> *This is what OSRM implements. All credit to the Project-OSRM contributors and to Geisberger et al. for the original algorithm.*